# DeltaNet算子实践

本 Notebook聚焦DeltaNet算子的完整实践路径：先建立公式直觉并对应到实现调用链，再通过最小可运行示例验证前向与反向数值一致性，最后给出与 torch 参考实现的性能对比结果，帮助你同时掌握“怎么用、怎么算、快多少”。

---

## 1. 环境准备


In [11]:
import io
import os
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

import torch
import torch.nn.functional as F


@contextmanager
def suppress_triton_tune_warning():
    needle = "Please DO NOT tune args"
    out_buf, err_buf = io.StringIO(), io.StringIO()
    with redirect_stdout(out_buf), redirect_stderr(err_buf):
        yield

    def _replay_without_tune_warning(buf, target_stream):
        text = buf.getvalue()
        if not text:
            return
        kept = [line for line in text.splitlines(True) if needle not in line]
        if kept:
            target_stream.write("".join(kept))
            target_stream.flush()

    _replay_without_tune_warning(out_buf, sys.stdout)
    _replay_without_tune_warning(err_buf, sys.stderr)


def _find_sample_root(start: Path) -> Path:
    for base in [start, *start.parents]:
        if (base / "internal" / "tla" / "nlp" / "tla-nlp-frame").is_dir():
            return base
    return start


cwd = Path.cwd().resolve()
sample_root = _find_sample_root(cwd)
internal = sample_root / "internal"
frame = internal / "tla" / "nlp" / "tla-nlp-frame"
if str(frame) not in sys.path:
    sys.path.insert(0, str(frame))

try:
    os.chdir(sample_root)
except OSError:
    pass

print("sample_root:", sample_root)

try:
    import torch_npu  # noqa: F401
    print("import torch_npu: OK")
except Exception as e:
    print("import torch_npu: SKIP/FAILED", e)

device = "npu" if hasattr(torch, "npu") and torch.npu.is_available() else "cpu"
print("device:", device)

from tla.ops.delta_rule import chunk_delta_rule, fused_recurrent_delta_rule

_dn_path = internal / "tla" / "torch" / "delta_net.py"
_spec = spec_from_file_location("_delta_net_torch", _dn_path)
_mod = module_from_spec(_spec)
_spec.loader.exec_module(_mod)
chunk_batched_delta_rule_forward_multi = _mod.chunk_batched_delta_rule_forward_multi

print("import delta_rule ops: OK")
print("import chunk_batched_delta_rule_forward_multi: OK")

project_root: /home/ma-user/work/Ascend-TLA
import torch_npu: OK
device: npu
import delta_rule ops: OK
import chunk_batched_delta_rule_forward_multi: OK


---

## 2. DeltaNet 算子

### 2.1 数学形式、实现与函数调用关系

本节先建立 DeltaNet 的更新直觉，再对照实现调用链，帮助你把公式与代码一一对应起来。

长序列下，标准 Softmax 注意力往往受限于构造 $N\times N$ 相似度矩阵的开销。因果设定下，DeltaNet 用记忆矩阵 $\mathbf{S}_t$ **递推更新**，通过秩一修正避免显式存储全矩阵，并在键方向支持「覆写式」联想更新。

**DeltaNet（Delta Rule）的关键特点**可概括为三点：

1. **关联式覆写（associative write / delta update）**：在键 $\mathbf{k}_t$ 方向上，用「旧读出 $\mathbf{S}_{t-1}\mathbf{k}_t$」与「新值 $\mathbf{v}_t$」经门控得到新向量，再对 $\mathbf{S}$ 做**减旧外积、加新外积**的秩一更新；相对仅累加外积的记忆形式，DeltaNet 显式建模**在同一键子空间上覆盖旧联想**的能力。
2. **标量门控 $\beta_t$**：每一步控制新值相对旧读出的比例，从而调节**写入强度**与**保留历史**；许多注意力形式将写入强度隐含在核或相似度中，DeltaNet 使用显式 $\beta_t$ 直接作用在记忆更新上。
3. **仍保持线性时间潜力**：递推仍围绕矩阵–向量乘与秩一更新展开；工程上通过 chunk 化与 Triton kernel 组织并行计算。


#### Delta Net递推形式

记记忆矩阵 $\mathbf{S}_{t-1} \in \mathbb{R}^{d \times d}$，单步输入为查询、键、值 $\mathbf{q}_t, \mathbf{k}_t, \mathbf{v}_t \in \mathbb{R}^d$ 以及门控 $\beta_t \in (0,1)$。旧读出与新写入为：

$$
\mathbf{v}_t^{\mathrm{old}} = \mathbf{S}_{t-1}\mathbf{k}_t, \qquad
\mathbf{v}_t^{\mathrm{new}} = \beta_t \mathbf{v}_t + (1-\beta_t)\,\mathbf{v}_t^{\mathrm{old}}
$$

状态按 Delta 规则更新，输出为：

$$
\mathbf{S}_t = \mathbf{S}_{t-1} - \mathbf{v}_t^{\mathrm{old}}\mathbf{k}_t^\top + \mathbf{v}_t^{\mathrm{new}}\mathbf{k}_t^\top
= \mathbf{S}_{t-1} + (\mathbf{v}_t^{\mathrm{new}} - \mathbf{v}_t^{\mathrm{old}})\mathbf{k}_t^\top
$$

$$
\mathbf{o}_t = \mathbf{S}_t \mathbf{q}_t
$$

即：在键方向 $\mathbf{k}_t$ 上把“旧值读出”替换为门控后的“新值”，再与 $\mathbf{q}_t$ 做右乘得到输出。

#### 调用关系图


（函数调用关系示意图可参考上游仓库 `Ascend-TLA` 中 `quick_start` 配图。）




#### 函数职责

下面这组函数可以看作从“模型层输入”到“kernel 执行”再到“反向回传”的完整数据路径。

- `custom_gpt2_attention_forward(...)`：在模型层构造整段序列的 $\mathbf{q}_t,\mathbf{k}_t,\mathbf{v}_t$ 和门控 $\beta_t$，并把它们送入 DeltaRule 路径。
- `eager_attention_forward(...)`：做输入整理（shape/归一化/`beta` 约束），确保送进算子的就是递推里对应的 $q,k,v,\beta$。
- `chunk_delta_rule(...)`：作为统一入口接收整段 $(q,k,v,\beta)$，并交给 Autograd Function 执行。
- `ChunkDeltaRuleFunction.forward(...)`：前向调度与上下文保存层，负责把后续反向要用到的中间量缓存起来。
- `chunk_delta_rule_fwd(...)`：前向主流程，把“计算 $\mathbf{v}_t^{old}/\mathbf{v}_t^{new}$、更新 $\mathbf{S}_t$、得到 $\mathbf{o}_t$”拆成三个阶段执行。
- `prepare_wy_repr_fwd(...)`：先在 chunk 内准备辅助表示（`w/u/A`），用于高效近似/等价实现每步的旧值读出与新值写入。
- `chunk_gated_delta_rule_fwd_h(...)`：核心状态更新阶段，对应递推中的
  $\mathbf{S}_{t-1}\rightarrow\mathbf{S}_t$，并得到与 $\mathbf{v}_t^{new}$ 对应的更新后值表示。
- `chunk_fwd_o(...)`：对应 $\mathbf{o}_t=\mathbf{S}_t\mathbf{q}_t$，把查询与更新后的状态信息合成为最终输出。
- `ChunkDeltaRuleFunction.backward(...)`：反向总调度，组织 `chunk_delta_rule_bwd(...)` 计算所有梯度。
- `chunk_delta_rule_bwd(...)`：按前向同样的数据流回传，得到对 $q,k,v,\beta$（以及可选初始状态）的梯度。
- `recompute_w_u_fwd(...)`：反向中的重计算步骤，用来恢复前向辅助量，减少前向显存缓存。

#### Triton实现思路

- 从公式看，DeltaNet 每步都要做三件事：
  1) 用 $\mathbf{S}_{t-1}$ 和 $\mathbf{k}_t$ 得到 $\mathbf{v}_t^{old}$；
  2) 用 $\beta_t$ 与 $\mathbf{v}_t$ 混合得到 $\mathbf{v}_t^{new}$，再更新 $\mathbf{S}_t$；
  3) 用 $\mathbf{S}_t$ 与 $\mathbf{q}_t$ 得到 $\mathbf{o}_t$。
- 工程实现没有逐 token 独立调度，而是按 chunk（常见 `BT=64`）批量处理，把这三类计算映射到“表示准备 → 状态更新 → 输出合成”三段 kernel 流水。
- 其中状态更新阶段（`chunk_gated_delta_rule_fwd_h(...)`）是融合重点：把与 $\mathbf{v}_t^{old}$、$\mathbf{v}_t^{new}$ 和 $\mathbf{S}_t$ 更新相关的计算集中处理，减少中间张量往返。
- 反向沿同一语义链路回传：对应 $\mathbf{o}_t$、$\mathbf{S}_t$、$\mathbf{v}_t^{new}$、$\mathbf{v}_t^{old}$ 逐层把梯度传回到 $q,k,v,\beta$，并通过 `recompute_w_u_fwd(...)` 做“算力换显存”的折中。




### 2.2 调用示例

本节目标是验证 Triton 实现与 torch 参考实现在前向输出与梯度上的一致性。

In [12]:
B, T, H, D = 1, 128, 1, 64
assert T % 64 == 0, "参考实现 chunk_batched_delta_rule_forward_multi 要求 T 能被 C=64 整除"

if device == "npu":
    dtype = torch.float16
elif hasattr(torch, "bfloat16") and (
    device == "cuda" or getattr(torch.cpu, "is_bf16_supported", lambda: False)()
):
    dtype = torch.bfloat16
else:
    dtype = torch.float16

torch.manual_seed(42)
q = torch.randn(B, T, H, D, dtype=dtype, device=device, requires_grad=True)
k = torch.randn(B, T, H, D, dtype=dtype, device=device, requires_grad=True)
v = torch.randn(B, T, H, D, dtype=dtype, device=device, requires_grad=True)
beta = torch.randn(B, T, H, dtype=dtype, device=device).sigmoid().requires_grad_(True)
h0 = torch.randn(B, H, D, D, dtype=torch.float32, device=device, requires_grad=True)
do = torch.randn_like(v)
dht = torch.randn_like(h0)

with suppress_triton_tune_warning():
    tri, tri_ht = chunk_delta_rule(
        q=F.normalize(q.clone(), p=2, dim=-1),
        k=F.normalize(k.clone(), p=2, dim=-1),
        v=v.clone(),
        beta=beta.clone(),
        scale=1.0,
        output_final_state=True,
        initial_state=h0.clone(),
        use_qk_l2norm_in_kernel=False,
    )
((tri * do).sum() + (tri_ht * dht).sum()).backward(retain_graph=True)
tri_dq, tri_dk, tri_dv, tri_dbeta, tri_dh0 = q.grad, k.grad, v.grad, beta.grad, h0.grad
q.grad = k.grad = v.grad = beta.grad = h0.grad = None

with suppress_triton_tune_warning():
    ref, ref_ht = chunk_batched_delta_rule_forward_multi(
        Q=F.normalize(q.clone(), p=2, dim=-1),
        K=F.normalize(k.clone(), p=2, dim=-1),
        V=v.clone(),
        beta=beta.clone(),
        C=64,
        initial_state=h0.clone(),
    )
((ref * do).sum() + (ref_ht * dht).sum()).backward()
ref_dq, ref_dk, ref_dv, ref_dbeta, ref_dh0 = q.grad, k.grad, v.grad, beta.grad, h0.grad

def _ok(name, a, b, atol):
    close = torch.allclose(a, b, atol=atol, rtol=0)
    print(f"{name} allclose (atol={atol}):", close)
    return close

assert _ok("o", ref, tri, 6e-3)
assert _ok("ht", ref_ht, tri_ht, 6e-3)
assert _ok("dq", ref_dq, tri_dq, 8e-3)
assert _ok("dk", ref_dk, tri_dk, 8e-3)
assert _ok("dv", ref_dv, tri_dv, 8e-3)
assert _ok("dh0", ref_dh0, tri_dh0, 8e-3)
print("[PASS] DeltaNet chunk forward + backward vs torch reference")

o allclose (atol=0.006): True
ht allclose (atol=0.006): True
dq allclose (atol=0.008): True
dk allclose (atol=0.008): True
dv allclose (atol=0.008): True
dh0 allclose (atol=0.008): True
[PASS] DeltaNet chunk forward + backward vs torch reference


### 2.3 性能测试

数值一致性通过后，本节只关注性能：对比 **PyTorch 参考** `chunk_batched_delta_rule_forward_multi` 与 **Triton 实现** `chunk_delta_rule` 的前向与反向耗时。
这里固定 `B/H/D`、改变序列长度 `T`，用于观察上下文长度增长时的加速趋势。计时使用 `triton.testing.do_bench`（典型的微基准计时方式）。加速比：

$$
\text{Speedup}=\frac{T_{\text{torch}}}{T_{\text{triton}}}
$$

下面给出测试代码

In [13]:
import triton


def _delta_dtype():
    if device == "npu":
        return torch.float16
    if hasattr(torch, "bfloat16") and (
        device == "cuda" or getattr(torch.cpu, "is_bf16_supported", lambda: False)()
    ):
        return torch.bfloat16
    return torch.float16


def benchmark_delta_torch_vs_triton(B, T, H, D, device=None):
    dev = device or globals().get("device", "cpu")
    dtype = _delta_dtype()
    if T % 64 != 0:
        print(f"[SKIP] T={T} 不能被参考实现 C=64 整除")
        return

    torch.manual_seed(0)
    q = torch.randn(B, T, H, D, dtype=dtype, device=dev, requires_grad=True)
    k = torch.randn(B, T, H, D, dtype=dtype, device=dev, requires_grad=True)
    v = torch.randn(B, T, H, D, dtype=dtype, device=dev, requires_grad=True)
    beta = torch.rand(B, T, H, dtype=dtype, device=dev).sigmoid().requires_grad_(True)
    h0 = torch.randn(B, H, D, D, dtype=torch.float32, device=dev, requires_grad=True)

    qn = F.normalize(q.detach(), p=2, dim=-1).requires_grad_(True)
    kn = F.normalize(k.detach(), p=2, dim=-1).requires_grad_(True)
    v_ = v.detach().requires_grad_(True)
    beta_ = beta.detach().requires_grad_(True)
    h0_ = h0.detach().requires_grad_(True)

    # 与参与 forward 的张量严格同形；ht 常为 float32，反向里 dht 对齐到 ht
    do = torch.randn_like(v_)
    dht_base = torch.randn_like(h0_)

    def run_triton():
        with suppress_triton_tune_warning():
            o, ht = chunk_delta_rule(
                q=qn, k=kn, v=v_, beta=beta_,
                scale=1.0, initial_state=h0_, output_final_state=True,
                use_qk_l2norm_in_kernel=False,
            )
        return o, ht

    def run_torch():
        with suppress_triton_tune_warning():
            o, ht = chunk_batched_delta_rule_forward_multi(
                Q=qn, K=kn, V=v_, beta=beta_, C=64, initial_state=h0_,
            )
        return o, ht

    exp_o = (B, T, H, D)
    exp_ht = (B, H, D, D)
    o_t, ht_t = run_torch()
    o_n, ht_n = run_triton()
    assert o_t.shape == exp_o, f"torch o.shape={o_t.shape}, expected {exp_o}"
    assert o_n.shape == exp_o, f"triton o.shape={o_n.shape}, expected {exp_o}"
    assert ht_t.shape == exp_ht, f"torch ht.shape={ht_t.shape}, expected {exp_ht}"
    assert ht_n.shape == exp_ht, f"triton ht.shape={ht_n.shape}, expected {exp_ht}"
    assert do.shape == o_t.shape, f"do.shape={do.shape}, o.shape={o_t.shape}"
    dht_t = dht_base.to(dtype=ht_t.dtype)
    dht_n = dht_base.to(dtype=ht_n.dtype)
    assert dht_t.shape == ht_t.shape
    # 注意：必须是「两个标量 loss 相加」，不能写成 (o*do + ht*dht).sum()——
    # 后者会先对四维张量做逐元素加，(B,T,H,D) 与 (B,H,D,D) 会错误广播并在 H/D 维报 8 vs 64。
    loss_t = (o_t * do).sum() + (ht_t * dht_t).sum()
    loss_t.backward(retain_graph=False)
    qn.grad = kn.grad = v_.grad = beta_.grad = h0_.grad = None
    loss_n = (o_n * do).sum() + (ht_n * dht_n).sum()
    loss_n.backward()
    qn.grad = kn.grad = v_.grad = beta_.grad = h0_.grad = None
    print("[sanity] shapes OK; torch + triton backward once each OK")

    num_runs = 10
    sum_torch_fwd = sum_tri_fwd = sum_torch_bwd = sum_tri_bwd = 0.0

    for i in range(num_runs):
        ms_torch_fwd = triton.testing.do_bench(lambda: run_torch()[0])
        ms_tri_fwd = triton.testing.do_bench(lambda: run_triton()[0])

        def bench_bwd(fn, dht_like):
            def _inner():
                qn.grad = kn.grad = v_.grad = beta_.grad = h0_.grad = None
                o, ht = fn()
                dht = dht_like.to(dtype=ht.dtype)
                ((o * do).sum() + (ht * dht).sum()).backward()

            return _inner

        ms_torch_bwd = triton.testing.do_bench(bench_bwd(run_torch, dht_base))
        ms_tri_bwd = triton.testing.do_bench(bench_bwd(run_triton, dht_base))

        sum_torch_fwd += ms_torch_fwd
        sum_tri_fwd += ms_tri_fwd
        sum_torch_bwd += ms_torch_bwd
        sum_tri_bwd += ms_tri_bwd
        print(
            f"Iter {i+1}/{num_runs}: Fwd {ms_torch_fwd/ms_tri_fwd:.2f}x | Bwd {ms_torch_bwd/ms_tri_bwd:.2f}x"
        )

    avg_tf = sum_torch_fwd / num_runs
    avg_tr = sum_tri_fwd / num_runs
    avg_tb = sum_torch_bwd / num_runs
    avg_trb = sum_tri_bwd / num_runs
    print("\n" + "=" * 80)
    print(f"DeltaNet chunk | B={B}, T={T}, H={H}, D={D}, dtype={dtype}")
    print("-" * 80)
    print(f"Forward : torch={avg_tf:.4f} ms, triton={avg_tr:.4f} ms, speedup={avg_tf/avg_tr:.2f}x")
    print(f"Backward: torch={avg_tb:.4f} ms, triton={avg_trb:.4f} ms, speedup={avg_tb/avg_trb:.2f}x")
    tot_t = avg_tf + avg_tb
    tot_r = avg_tr + avg_trb
    print(f"Total   : torch={tot_t:.4f} ms, triton={tot_r:.4f} ms, speedup={tot_t/tot_r:.2f}x")
    print("=" * 80)



for T in [512, 1024, 2048, 4096]:
    benchmark_delta_torch_vs_triton(B=2, T=T, H=8, D=64)

[sanity] shapes OK; torch + triton backward once each OK
Iter 1/10: Fwd 14.25x | Bwd 16.58x
Iter 2/10: Fwd 16.55x | Bwd 17.22x
Iter 3/10: Fwd 15.40x | Bwd 19.19x
Iter 4/10: Fwd 14.99x | Bwd 17.47x
Iter 5/10: Fwd 14.86x | Bwd 17.44x
Iter 6/10: Fwd 14.94x | Bwd 18.07x
Iter 7/10: Fwd 15.50x | Bwd 17.44x
Iter 8/10: Fwd 15.35x | Bwd 17.43x
Iter 9/10: Fwd 14.95x | Bwd 18.48x
Iter 10/10: Fwd 15.20x | Bwd 17.69x

DeltaNet chunk | B=2, T=512, H=8, D=64, dtype=torch.float16
--------------------------------------------------------------------------------
Forward : torch=28.4119 ms, triton=1.8693 ms, speedup=15.20x
Backward: torch=86.2929 ms, triton=4.8752 ms, speedup=17.70x
Total   : torch=114.7048 ms, triton=6.7446 ms, speedup=17.01x
[sanity] shapes OK; torch + triton backward once each OK
Iter 1/10: Fwd 10.14x | Bwd 11.76x
Iter 2/10: Fwd 9.71x | Bwd 11.36x
Iter 3/10: Fwd 10.00x | Bwd 11.63x
Iter 4/10: Fwd 9.70x | Bwd 11.64x
Iter 5/10: Fwd 11.26x | Bwd 10.84x
Iter 6/10: Fwd 9.37x | Bwd 11.61x
It

## 🚀 Kernel 性能对比（torch_npu vs Triton）

下面汇总了Delta Net算子的性能对比结果。

**测试设备：** Ascend 910B（1 卡）  
**测试配置：** $B=2,\ H=8,\ D=64$


| 序列长度 ($L$) | 阶段 | `torch_npu` (ms) | `Triton` (ms) | 加速比 |
| :---: | :--- | :---: | :---: | :---: |
| **512** | 前向 (Forward) | 30.0186 | 1.8664 | <span style="color: green">16.08x</span> |
|  | 反向 (Backward) | 85.8008 | 4.8144 | <span style="color: green">17.82x</span> |
|  | **总计 (Total)** | **115.8194** | **6.6809** | <span style="color: green">**17.34x**</span> |
|  |  |  |  |  |
| **1024** | 前向 (Forward) | 35.6055 | 3.4968 | <span style="color: green">10.18x</span> |
|  | 反向 (Backward) | 108.3010 | 9.1353 | <span style="color: green">11.86x</span> |
|  | **总计 (Total)** | **143.9065** | **12.6321** | <span style="color: green">**11.39x**</span> |
|  |  |  |  |  |
| **2048** | 前向 (Forward) | 48.9496 | 6.7131 | <span style="color: green">7.29x</span> |
|  | 反向 (Backward) | 148.9113 | 17.5868 | <span style="color: green">8.47x</span> |
|  | **总计 (Total)** | **197.8609** | **24.2999** | <span style="color: green">**8.14x**</span> |
|  |  |  |  |  |
| **4096** | 前向 (Forward) | 77.0060 | 13.2485 | <span style="color: green">5.81x</span> |
|  | 反向 (Backward) | 233.0150 | 34.8892 | <span style="color: green">6.68x</span> |
|  | **总计 (Total)** | **310.0209** | **48.1377** | <span style="color: green">**6.44x**</span> |

上表为 **10 次 `do_bench` 平均** 的汇总行数值；各轮迭代中 Fwd/Bwd 相对加速约在 **5.5x–18.6x** 区间。
通常随着 `T` 增大，加速比会有所回落但仍保持明显优势；若你本地重跑与表内略有偏差，以本地输出为准。